# Module 2 — everything, on one GPU

Runs both exercise tasks end to end and prints both report tables.

| | Task 1 — Polyp characterization | Task 2 — Semantic segmentation |
|---|---|---|
| Deep-dive model | **MobileNetV3-Large** | **Attention U-Net** |
| Comparison 2 | ResNet18 | U-Net |
| Comparison 3 | EfficientNet-B0 | U-Net++ |

## How to run this

1. **Runtime -> Change runtime type -> T4 GPU -> Save**
2. **Runtime -> Run all**
3. When the second cell shows a **"Choose Files"** button, pick **both** zips from your
   Mac at the same time (hold Cmd to select two):
   - `module2_code.zip`
   - `module2_data.zip`
4. Wait. About 50 minutes.

No Google Drive, no folder paths, no accounts. The membrane data for Task 2 downloads
itself. At the end your results download straight back to your Mac.

## 0. Setup

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "NO GPU — set Runtime > Change runtime type > T4 GPU")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## Upload your two zips

Run this cell. A **"Choose Files"** button appears — pick **both** at once:
`module2_code.zip` and `module2_data.zip`.

The 43 MB one takes a few minutes. Wait for the cell to finish before going on.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from google.colab import files

SEEDS = [27]   # [27, 1, 2] gives mean +- std, and takes ~3x longer

WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
os.chdir("/content")

uploaded = files.upload()
for name in uploaded:
    with zipfile.ZipFile(f"/content/{name}") as z:
        z.extractall(WORK)
    print("unpacked", name)

os.chdir(WORK)
sys.path.insert(0, WORK)
POLYP_ROOT = WORK

if not os.path.isdir(f"{WORK}/src"):
    raise FileNotFoundError("no src/ — you did not upload module2_code.zip")

RUN_TASK1 = os.path.isdir(f"{WORK}/PCData")
RUN_TASK2 = True

print("\ncode:", sorted(os.listdir("src")))
if RUN_TASK1:
    print("polyp images:",
          len(os.listdir("PCData/Neoplasia")), "neoplasia,",
          len(os.listdir("PCData/Non-neoplasia")), "non-neoplasia")
else:
    print("no PCData/ — skipping Task 1, Task 2 will still run")

In [ ]:
# The membrane dataset is public — fetch it here so nothing has to be uploaded for Task 2.
if RUN_TASK2 and not os.path.isdir("data/train/images"):
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    "https://github.com/zhixuhao/unet.git", "/tmp/unet_repo"], check=True)
    src = "/tmp/unet_repo/data/membrane"
    for dst, pattern in [("data/train/images", "train/image"),
                         ("data/train/labels", "train/label")]:
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(f"{src}/{pattern}"):
            shutil.copy(f"{src}/{pattern}/{f}", dst)
    os.makedirs("data/test", exist_ok=True)
    for f in os.listdir(f"{src}/test"):
        if not f.endswith("_predict.png"):
            shutil.copy(f"{src}/test/{f}", "data/test")

if RUN_TASK2:
    print("membrane:", len(os.listdir("data/train/images")), "images,",
          len(os.listdir("data/train/labels")), "labels,",
          len(os.listdir("data/test")), "unlabelled test")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from src.attention_unet import AttentionUNet, UNet
from src.losses import LOSSES
from src.membrane import load_membrane
from src.metrics import best_threshold, binary_metrics, plot_confusion, plot_roc
from src.results import results_table, save_run
from src.seed import get_device, set_seed
from src.unetpp import UNetPlusPlus

import src.polyp as polyp
import src.segmentation as seg

device = get_device()
print("device:", device)

---
# Task 1 — Polyp Characterization

Three models, identical training procedure, so the comparison is about the architecture
and nothing else. Early stopping patience 5; the decision threshold is chosen on
**validation** (Youden's J) and then applied unchanged to test.

In [ ]:
POLYP_MODELS = ["mobilenet_v3_large", "resnet18", "efficientnet_b0"]
POLYP_CFG = dict(image_size=224, batch_size=32, lr=1e-4, n_epochs=50,
                 patience=5, pretrained=True)

polyp_results, polyp_curves = {}, {}

if RUN_TASK1:
    train_loader, val_loader, test_loader = polyp.load_polyp_splits(
        root=POLYP_ROOT, image_size=POLYP_CFG["image_size"],
        batch_size=POLYP_CFG["batch_size"], augment=True, num_workers=2)

In [ ]:
if RUN_TASK1:
    for name in POLYP_MODELS:
        per_seed = []
        for seed in SEEDS:
            set_seed(seed)
            print(f"\n=== {name}  seed {seed} ===")
            model = polyp.build_model(name, pretrained=POLYP_CFG["pretrained"]).to(device)
            criterion = nn.BCEWithLogitsLoss(
                pos_weight=polyp.pos_weight_from(train_loader, device))
            optimizer = torch.optim.Adam(model.parameters(), lr=POLYP_CFG["lr"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="min", factor=0.5, patience=2)

            history = polyp.fit(model, train_loader, val_loader, criterion, optimizer,
                                device, n_epochs=POLYP_CFG["n_epochs"],
                                patience=POLYP_CFG["patience"], scheduler=scheduler,
                                checkpoint_path=f"/content/work/{name}_s{seed}.pt")

            _, y_val, p_val, _ = polyp.evaluate(model, val_loader, criterion, device)
            threshold = best_threshold(y_val, p_val)
            _, y_test, p_test, _ = polyp.evaluate(model, test_loader, criterion, device)
            metrics = binary_metrics(y_test, p_test, threshold)

            per_seed.append(metrics)
            polyp_curves[name] = (y_test, p_test, threshold)
            save_run(tag=f"{name}_s{seed}", config={**POLYP_CFG, "model": name,
                     "seed": seed, "threshold": threshold},
                     metrics=metrics, history=history, model=model)
            print({k: round(v, 3) for k, v in metrics.items()})

        polyp_results[name] = {
            k: float(np.mean([m[k] for m in per_seed])) for k in per_seed[0]}
        if len(SEEDS) > 1:
            polyp_results[name].update({
                f"{k}_std": float(np.std([m[k] for m in per_seed])) for k in per_seed[0]})

### Table 1 — paste this into the report

In [ ]:
if RUN_TASK1:
    print(results_table(polyp_results,
                        ["accuracy", "sensitivity", "specificity", "f1", "auc"]))

### ROC curves and confusion matrices (report requirement 4)

In [ ]:
if RUN_TASK1:
    fig, ax = plt.subplots(figsize=(6, 6))
    for name, (y, p, _) in polyp_curves.items():
        plot_roc(y, p, label=name, ax=ax)
    ax.set_title("ROC — all three models")
    fig.tight_layout(); fig.savefig("task1_roc.png", dpi=150)

    fig, axes = plt.subplots(1, len(polyp_curves), figsize=(5 * len(polyp_curves), 5))
    for ax, (name, (y, p, thr)) in zip(np.atleast_1d(axes), polyp_curves.items()):
        plot_confusion(y, (p >= thr).astype(int),
                       class_names=["non-neoplasia", "neoplasia"], ax=ax)
        ax.set_title(name, fontsize=10)
    fig.tight_layout(); fig.savefig("task1_confusion.png", dpi=150)

---
# Task 2 — Semantic Segmentation

The three models isolate one change each:

- **U-Net** — the baseline.
- **Attention U-Net** — baseline **+ attention gates on the skips** (+0.35M params, ~1%).
- **U-Net++** — baseline **+ nested dense skip pathways**.

So if one wins you can say *why*, rather than guessing. Loss is Dice+BCE for all three;
the loss ablation is a separate section further down.

The official ISBI test folder has **no ground truth**, so metrics come from a 20/5/5
split of the 30 labelled images. State this in the report.

In [ ]:
SEG_MODELS = {
    "attention_unet": lambda: AttentionUNet(base=64),
    "unet":           lambda: UNet(base=64),
    "unetpp":         lambda: UNetPlusPlus(base=64),
}
SEG_CFG = dict(crop_size=256, batch_size=4, lr=1e-4, n_epochs=40,
               patience=5, repeats=8, loss="dice_bce", splits=(20, 5, 5))

seg_results = {}

if RUN_TASK2:
    for name, factory in SEG_MODELS.items():
        per_seed = []
        for seed in SEEDS:
            set_seed(seed)
            print(f"\n=== {name}  seed {seed} ===")
            tr, va, te = load_membrane(root="data", crop_size=SEG_CFG["crop_size"],
                                       batch_size=SEG_CFG["batch_size"],
                                       splits=SEG_CFG["splits"],
                                       repeats=SEG_CFG["repeats"], seed=seed)
            model = factory().to(device)
            criterion = LOSSES[SEG_CFG["loss"]]()
            optimizer = torch.optim.Adam(model.parameters(), lr=SEG_CFG["lr"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="min", factor=0.5, patience=3)

            history = seg.fit(model, tr, va, criterion, optimizer, device,
                              n_epochs=SEG_CFG["n_epochs"], patience=SEG_CFG["patience"],
                              scheduler=scheduler,
                              checkpoint_path=f"/content/work/{name}_s{seed}.pt")
            metrics, _ = seg.evaluate(model, te, criterion, device)
            per_seed.append(metrics)
            save_run(tag=f"{name}_{SEG_CFG['loss']}_s{seed}",
                     config={**SEG_CFG, "model": name, "seed": seed},
                     metrics=metrics, history=history, model=model)
            print({k: round(v, 4) for k, v in metrics.items()})

        seg_results[name] = {k: float(np.mean([m[k] for m in per_seed]))
                             for k in per_seed[0]}
        if len(SEEDS) > 1:
            seg_results[name].update({f"{k}_std": float(np.std([m[k] for m in per_seed]))
                                      for k in per_seed[0]})
        if name == "attention_unet":
            best_attention_model, attention_test_loader = model, te

### Table 2 — paste this into the report

In [ ]:
if RUN_TASK2:
    print(results_table(seg_results,
                        ["jaccard", "f1", "recall", "precision", "accuracy"], "{:.4f}"))

### Qualitative results (report requirement 5)

In [ ]:
if RUN_TASK2:
    model = best_attention_model.eval()
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    with torch.no_grad():
        for col, (image, mask) in zip(range(3), attention_test_loader):
            prob = torch.sigmoid(model(image.to(device)))[0, 0].cpu()
            for row, (img, title) in enumerate([
                    (image[0, 0], "original"), (mask[0, 0], "ground truth"),
                    (prob >= 0.5, "prediction")]):
                axes[row][col].imshow(img, cmap="gray")
                axes[row][col].set_title(title); axes[row][col].axis("off")
    fig.tight_layout(); fig.savefig("task2_qualitative.png", dpi=150)

### Attention maps

Not required, but it is the one figure a plain-U-Net report cannot produce, and it is
direct evidence for what you claim the gates do. Bright = the gate passed that region through.

In [ ]:
if RUN_TASK2:
    image, _ = next(iter(attention_test_loader))
    with torch.no_grad():
        _, attention = best_attention_model(image.to(device), return_attention=True)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(image[0, 0], cmap="gray"); axes[0].set_title("input")
    for ax, a in zip(axes[1:], attention):
        ax.imshow(a[0, 0].cpu(), cmap="inferno"); ax.set_title(f"gate @ {tuple(a.shape[-2:])}")
    for ax in axes: ax.axis("off")
    fig.tight_layout(); fig.savefig("task2_attention.png", dpi=150)

### Loss ablation (report requirement 6)

The exercise asks you to analyse "the effect of the selected loss function", so run it.
Attention U-Net, one seed, four losses.

Watch for this: **BCE usually wins on pixel accuracy and loses on Jaccard.** That single
contrast is the whole argument for why accuracy is the wrong metric for segmentation.

In [ ]:
LOSS_ABLATION = ["bce", "dice", "dice_bce", "iou_bce"]
ablation = {}

if RUN_TASK2:
    for loss_name in LOSS_ABLATION:
        set_seed(SEEDS[0])
        print(f"\n=== attention_unet / {loss_name} ===")
        tr, va, te = load_membrane(root="data", crop_size=SEG_CFG["crop_size"],
                                   batch_size=SEG_CFG["batch_size"],
                                   splits=SEG_CFG["splits"],
                                   repeats=SEG_CFG["repeats"], seed=SEEDS[0])
        model = AttentionUNet(base=64).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=SEG_CFG["lr"])
        history = seg.fit(model, tr, va, LOSSES[loss_name](), optimizer, device,
                          n_epochs=SEG_CFG["n_epochs"], patience=SEG_CFG["patience"],
                          checkpoint_path=f"/content/work/abl_{loss_name}.pt")
        metrics, _ = seg.evaluate(model, te, LOSSES[loss_name](), device)
        ablation[loss_name] = metrics
        save_run(tag=f"ablation_{loss_name}", config={**SEG_CFG, "loss": loss_name},
                 metrics=metrics, history=history)
        print({k: round(v, 4) for k, v in metrics.items()})

    print()
    print(results_table(ablation, ["jaccard", "f1", "recall", "precision", "accuracy"], "{:.4f}"))

---
## Download your results

This packs everything into one zip and downloads it to your Mac: the figures, plus
`results/` with one folder per run holding `config.json`, `metrics.json`,
`history.json` and `model.pt`.

In [ ]:
import datetime

out = "/content/out"
shutil.rmtree(out, ignore_errors=True)
os.makedirs(out)

if os.path.isdir(f"{WORK}/results"):
    shutil.copytree(f"{WORK}/results", f"{out}/results")
else:
    print("no results/ folder — did any training actually run?")

for png in [p for p in os.listdir(WORK) if p.endswith(".png")]:
    shutil.copy(f"{WORK}/{png}", out)

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
archive = shutil.make_archive(f"/content/module2_results_{stamp}", "zip", out)
print("packed:", sorted(os.listdir(out)))
files.download(archive)

---
## Reading your own results

**Task 1.** Report sensitivity and specificity, not just accuracy — the test set is
64% neoplasia, so "always neoplasia" already scores 64%. If a model has high accuracy
and low specificity, that is what it learned.

**Task 2.** Accuracy is the least useful column; background dominates. Argue from
Jaccard and Dice.

**Both.** With one seed, a difference of a point or two is noise. If you want to claim
model A beats model B, set `SEEDS = [27, 1, 2]` and report mean +- std. Roughly triples
the runtime.

**Split caveat for Task 1.** The split files are polyp-level, and 22 patients have
polyps in both train and test. Same colon, same capsule, same session. Mention it —
noticing it counts in your favour.